# Analisis de precios
Basado en la base de datos de [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios)

In [5]:
# =============================================================================
# PASO 1 (corregido): Descompresión robusta con búsqueda recursiva
# =============================================================================
# Cambio clave: buscamos los ZIPs internos de forma RECURSIVA (rglob en vez
# de glob), porque el ZIP externo puede tener una carpeta intermedia adentro.
# =============================================================================

import os
import re
import zipfile
from pathlib import Path
import pandas as pd

# -----------------------------------------------------------------------------
# 1.1) Configuración de rutas (igual que antes)
# -----------------------------------------------------------------------------
CONTENT_DIR = Path("/content")
WORK_DIR    = Path("/content/sepa_data")
WORK_DIR.mkdir(exist_ok=True)

patron_fecha = re.compile(r"^(\d{4}-\d{2}-\d{2})\.zip$")

zips_dia = [p for p in CONTENT_DIR.iterdir()
            if p.is_file() and patron_fecha.match(p.name)]

if not zips_dia:
    raise FileNotFoundError(
        "No se encontró ningún ZIP con formato AAAA-MM-DD.zip en /content/."
    )

zip_dia   = sorted(zips_dia)[-1]
fecha_str = patron_fecha.match(zip_dia.name).group(1)

print(f"📦 ZIP del día detectado: {zip_dia.name}")
print(f"📅 Fecha inferida:        {fecha_str}")

# -----------------------------------------------------------------------------
# 1.2) Descompresión del ZIP externo (idempotente)
# -----------------------------------------------------------------------------
dia_dir = WORK_DIR / fecha_str
dia_dir.mkdir(exist_ok=True)

# Descomprimimos si todavía no existen ZIPs internos en ningún nivel
if not list(dia_dir.rglob("*.zip")):
    print(f"⏳ Descomprimiendo {zip_dia.name} ...")
    with zipfile.ZipFile(zip_dia, "r") as z:
        z.extractall(dia_dir)
    print(f"✅ Listo. Contenido en: {dia_dir}")
else:
    print(f"ℹ️  Ya hay ZIPs internos descomprimidos en {dia_dir}, no re-descomprimo el externo.")

# -----------------------------------------------------------------------------
# 1.3) Descompresión de los ZIPs internos — BÚSQUEDA RECURSIVA
# -----------------------------------------------------------------------------
# Usamos rglob("*.zip") para encontrar los ZIPs de cadenas estén donde estén
# (en el nivel raíz o dentro de una carpeta intermedia como "2026-04-24/").
zips_internos = list(dia_dir.rglob("*.zip"))
print(f"\n📂 ZIPs internos encontrados (búsqueda recursiva): {len(zips_internos)}")

# Para mantener todo organizado, descomprimimos cada uno en una subcarpeta
# DENTRO DE dia_dir (no junto al zip), así las carpetas de cadenas quedan
# todas al mismo nivel, sin importar dónde estaba el zip original.
for zi in zips_internos:
    destino = dia_dir / zi.stem  # <nombre_del_zip>/ directamente bajo dia_dir
    if destino.exists() and any(destino.iterdir()):
        continue
    destino.mkdir(exist_ok=True)
    try:
        with zipfile.ZipFile(zi, "r") as z:
            z.extractall(destino)
    except zipfile.BadZipFile:
        print(f"⚠️  ZIP corrupto, lo salteo: {zi.name}")

# -----------------------------------------------------------------------------
# 1.4) Inventario: SOLO carpetas que realmente contienen los CSVs de SEPA
# -----------------------------------------------------------------------------
# Antes filtrábamos "cualquier carpeta dentro de dia_dir", lo cual incluía
# carpetas intermedias vacías. Ahora nos quedamos solo con las que tienen
# al menos uno de los CSVs esperados.
CSVS_ESPERADOS = {"comercio.csv", "sucursales.csv", "productos.csv"}

carpetas_cadena = []
for carpeta in dia_dir.rglob("*"):
    if not carpeta.is_dir():
        continue
    nombres_csv = {f.name for f in carpeta.glob("*.csv")}
    if nombres_csv & CSVS_ESPERADOS:   # intersección no vacía
        carpetas_cadena.append(carpeta)

print(f"🏬 Carpetas de cadenas con CSVs SEPA: {len(carpetas_cadena)}")

# Armamos el inventario
resumen = []
for carpeta in carpetas_cadena:
    archivos = {f.name: f.stat().st_size for f in carpeta.glob("*.csv")}
    resumen.append({
        "carpeta": carpeta.name,
        "tiene_comercio":   "comercio.csv"   in archivos,
        "tiene_sucursales": "sucursales.csv" in archivos,
        "tiene_productos":  "productos.csv"  in archivos,
        "peso_productos_MB": round(archivos.get("productos.csv", 0) / (1024*1024), 2),
        "ruta": str(carpeta),
    })

df_inventario = (pd.DataFrame(resumen)
                 .sort_values("peso_productos_MB", ascending=False)
                 .reset_index(drop=True))

print(f"\n📋 Inventario completo ({len(df_inventario)} cadenas):")
# Mostramos sin la columna 'ruta' para que sea más legible
print(df_inventario.drop(columns=["ruta"]).to_string(index=False))

# Chequeos rápidos de consistencia
n_completas = df_inventario[
    df_inventario["tiene_comercio"] &
    df_inventario["tiene_sucursales"] &
    df_inventario["tiene_productos"]
].shape[0]
print(f"\n✅ Cadenas con los 3 CSVs completos: {n_completas} / {len(df_inventario)}")

# Guardamos variables clave para el próximo paso
print(f"\n✅ Paso 1 completo. Variables disponibles:")
print(f"   - dia_dir         = {dia_dir}")
print(f"   - fecha_str       = {fecha_str}")
print(f"   - df_inventario   (DataFrame con {len(df_inventario)} filas)")

📦 ZIP del día detectado: 2026-04-24.zip
📅 Fecha inferida:        2026-04-24
ℹ️  Ya hay ZIPs internos descomprimidos en /content/sepa_data/2026-04-24, no re-descomprimo el externo.

📂 ZIPs internos encontrados (búsqueda recursiva): 20
🏬 Carpetas de cadenas con CSVs SEPA: 20

📋 Inventario completo (20 cadenas):
                                    carpeta  tiene_comercio  tiene_sucursales  tiene_productos  peso_productos_MB
sepa_1_comercio-sepa-15_2026-04-24_09-05-10            True              True             True             443.12
sepa_2_comercio-sepa-10_2026-04-24_01-05-08            True              True             True             390.08
sepa_2_comercio-sepa-11_2026-04-24_01-05-08            True              True             True             235.59
 sepa_1_comercio-sepa-9_2026-04-24_09-05-10            True              True             True             146.41
sepa_1_comercio-sepa-12_2026-04-24_09-05-10            True              True             True             105.53
 sepa

In [7]:
# =============================================================================
# PASO 2 (corregido): manejo de la línea "Última actualización" al final
# =============================================================================
# Los CSVs de SEPA terminan con una línea tipo:
#   "Última actualización: 2026-04-24T05:01:26-03:00"
# Esta línea rompe el parser de pandas. La solución: leer el archivo como
# texto, quedarnos solo con las líneas que son datos (empiezan con un dígito,
# ya que los id son numéricos), y recién ahí parsear con pandas.
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO

# -----------------------------------------------------------------------------
# 2.1) Dtypes (idénticos al intento anterior — los dejamos por claridad)
# -----------------------------------------------------------------------------
DTYPES_COMERCIO = {
    "id_comercio": "Int64",
    "id_bandera":  "Int64",
    "comercio_cuit": "string",
    "comercio_razon_social": "string",
    "comercio_bandera_nombre": "string",
    "comercio_bandera_url": "string",
    "comercio_version_sepa": "string",
}

DTYPES_SUCURSALES = {
    "id_comercio": "Int64",
    "id_bandera":  "Int64",
    "id_sucursal": "Int64",
    "sucursales_nombre": "string",
    "sucursales_tipo": "string",
    "sucursales_calle": "string",
    "sucursales_numero": "string",
    "sucursales_latitud": "float64",
    "sucursales_longitud": "float64",
    "sucursales_observaciones": "string",
    "sucursales_barrio": "string",
    "sucursales_codigo_postal": "string",
    "sucursales_localidad": "string",
    "sucursales_provincia": "string",
}

DTYPES_PRODUCTOS = {
    "id_comercio": "Int64",
    "id_bandera":  "Int64",
    "id_sucursal": "Int64",
    "id_producto": "string",
    "productos_ean": "string",
    "productos_descripcion": "string",
    "productos_cantidad_presentacion": "float64",
    "productos_unidad_medida_presentacion": "string",
    "productos_marca": "string",
    "productos_precio_lista": "float64",
    "productos_precio_referencia": "float64",
    "productos_cantidad_referencia": "float64",
    "productos_unidad_medida_referencia": "string",
    "productos_precio_unitario_promo1": "float64",
    "productos_leyenda_promo1": "string",
    "productos_precio_unitario_promo2": "float64",
    "productos_leyenda_promo2": "string",
}

# -----------------------------------------------------------------------------
# 2.2) Lector robusto: pre-filtra líneas antes de pasar a pandas
# -----------------------------------------------------------------------------
def leer_csv_sepa(ruta, dtypes):
    """
    Lee un CSV de SEPA quitando:
      1. Líneas como "Última actualización: ..." al final del archivo.
      2. Líneas vacías.
    Estrategia: leemos el texto, nos quedamos con el header (primera línea) +
    líneas que tengan al menos N-1 pipes (donde N = cantidad esperada de
    columnas). Esto es más robusto que filtrar por "empieza con dígito",
    porque tolera filas de datos que, por algún motivo, arranquen distinto.
    """
    # 1) Leemos el archivo como texto, probando dos encodings
    try:
        with open(ruta, "r", encoding="utf-8") as f:
            texto = f.read()
    except UnicodeDecodeError:
        with open(ruta, "r", encoding="latin-1") as f:
            texto = f.read()

    lineas = texto.splitlines()
    if not lineas:
        raise ValueError(f"Archivo vacío: {ruta}")

    # 2) La primera línea es el header; contamos cuántas columnas tiene
    header = lineas[0]
    n_cols_esperadas = header.count("|") + 1

    # 3) Filtramos: nos quedamos con el header y con las líneas que tienen
    #    al menos n_cols_esperadas - 1 pipes. Las líneas de "Última
    #    actualización" tienen 0 pipes y quedan descartadas automáticamente.
    #    Las líneas vacías también.
    lineas_limpias = [header]
    descartadas = 0
    for linea in lineas[1:]:
        if not linea.strip():
            descartadas += 1
            continue
        if linea.count("|") < n_cols_esperadas - 1:
            descartadas += 1
            continue
        lineas_limpias.append(linea)

    # 4) Recién ahora pasamos a pandas usando StringIO (evita reescribir el archivo)
    texto_limpio = "\n".join(lineas_limpias)
    df = pd.read_csv(
        StringIO(texto_limpio),
        sep="|",
        dtype=dtypes,
        on_bad_lines="skip",
        na_values=["", "N/A", "NA", "NaN"],
    )

    return df, descartadas

# -----------------------------------------------------------------------------
# 2.3) Carga de los tres CSVs de cada cadena
# -----------------------------------------------------------------------------
lista_comercio, lista_sucursales, lista_productos = [], [], []
total_descartadas = {"comercio": 0, "sucursales": 0, "productos": 0}

print("⏳ Cargando CSVs de todas las cadenas (con limpieza previa)...")
for i, fila in df_inventario.iterrows():
    carpeta = Path(fila["ruta"])
    nombre_cadena = fila["carpeta"]

    if fila["tiene_comercio"]:
        df, desc = leer_csv_sepa(carpeta / "comercio.csv", DTYPES_COMERCIO)
        df["cadena_folder"] = nombre_cadena
        lista_comercio.append(df)
        total_descartadas["comercio"] += desc

    if fila["tiene_sucursales"]:
        df, desc = leer_csv_sepa(carpeta / "sucursales.csv", DTYPES_SUCURSALES)
        df["cadena_folder"] = nombre_cadena
        lista_sucursales.append(df)
        total_descartadas["sucursales"] += desc

    if fila["tiene_productos"]:
        df, desc = leer_csv_sepa(carpeta / "productos.csv", DTYPES_PRODUCTOS)
        df["cadena_folder"] = nombre_cadena
        lista_productos.append(df)
        total_descartadas["productos"] += desc
        print(f"   [{i+1:2d}/{len(df_inventario)}] {nombre_cadena[:50]:50s} "
              f"→ {len(df):>9,} filas  (descartadas: {desc})")

# Consolidamos
df_comercio   = pd.concat(lista_comercio,   ignore_index=True)
df_sucursales = pd.concat(lista_sucursales, ignore_index=True)
df_productos  = pd.concat(lista_productos,  ignore_index=True)

del lista_comercio, lista_sucursales, lista_productos

print("\n✅ Carga completa.")
print(f"   Líneas descartadas totales: "
      f"comercio={total_descartadas['comercio']}, "
      f"sucursales={total_descartadas['sucursales']}, "
      f"productos={total_descartadas['productos']}")

# -----------------------------------------------------------------------------
# 2.4) Resumen dimensional
# -----------------------------------------------------------------------------
def resumen_df(nombre, df):
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"\n📊 {nombre}")
    print(f"   Filas:     {len(df):>12,}")
    print(f"   Columnas:  {df.shape[1]:>12}")
    print(f"   Memoria:   {mem_mb:>10.1f} MB")

resumen_df("df_comercio",   df_comercio)
resumen_df("df_sucursales", df_sucursales)
resumen_df("df_productos",  df_productos)

# -----------------------------------------------------------------------------
# 2.5) Chequeos de calidad
# -----------------------------------------------------------------------------
print("\n🔍 Chequeos de calidad:")

n_razon_social = df_comercio["comercio_razon_social"].nunique()
n_banderas     = df_comercio["comercio_bandera_nombre"].nunique()
print(f"   Razones sociales únicas: {n_razon_social}")
print(f"   Banderas (marcas) únicas: {n_banderas}")

top_prov = df_sucursales["sucursales_provincia"].value_counts().head(5)
print(f"\n   Sucursales por provincia (top 5):")
for prov, n in top_prov.items():
    print(f"      {prov}: {n:,}")

n_precio_nulo = df_productos["productos_precio_lista"].isna().sum()
n_precio_cero = (df_productos["productos_precio_lista"] == 0).sum()
n_precio_neg  = (df_productos["productos_precio_lista"] < 0).sum()
print(f"\n   Productos con precio_lista nulo:     {n_precio_nulo:>10,}")
print(f"   Productos con precio_lista == 0:     {n_precio_cero:>10,}")
print(f"   Productos con precio_lista negativo: {n_precio_neg:>10,}")

n_ean_unicos = df_productos["productos_ean"].nunique()
print(f"\n   EANs únicos en toda la base: {n_ean_unicos:,}")

print("\n✅ Paso 2 completo. Variables disponibles:")
print("   - df_comercio, df_sucursales, df_productos")

⏳ Cargando CSVs de todas las cadenas (con limpieza previa)...
   [ 1/20] sepa_1_comercio-sepa-15_2026-04-24_09-05-10        → 4,201,667 filas  (descartadas: 2)
   [ 2/20] sepa_2_comercio-sepa-10_2026-04-24_01-05-08        → 3,605,280 filas  (descartadas: 2)
   [ 3/20] sepa_2_comercio-sepa-11_2026-04-24_01-05-08        → 2,794,161 filas  (descartadas: 2)
   [ 4/20] sepa_1_comercio-sepa-9_2026-04-24_09-05-10         → 1,318,219 filas  (descartadas: 2)
   [ 5/20] sepa_1_comercio-sepa-12_2026-04-24_09-05-10        → 1,017,148 filas  (descartadas: 2)
   [ 6/20] sepa_2_comercio-sepa-2_2026-04-24_01-05-08         →   879,719 filas  (descartadas: 2)
   [ 7/20] sepa_1_comercio-sepa-13_2026-04-24_09-05-10        →   690,553 filas  (descartadas: 2)
   [ 8/20] sepa_1_comercio-sepa-24_2026-04-24_09-05-10        →   372,502 filas  (descartadas: 2)
   [ 9/20] sepa_1_comercio-sepa-21_2026-04-24_09-05-10        →    91,685 filas  (descartadas: 2)
   [10/20] sepa_1_comercio-sepa-16_2026-04-24_09-05-10  

In [8]:
# =============================================================================
# PASO 2.5: Diagnóstico del campo EAN y optimización de memoria
# =============================================================================
# Dos objetivos:
#   A) Entender por qué solo hay 2 EANs únicos (sospecha: parseo incorrecto).
#   B) Reducir memoria de df_productos (hoy 9.3 GB) convirtiendo columnas
#      repetitivas a 'category' y achicando dtypes numéricos.
# =============================================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 2.5.A) Diagnóstico del EAN
# -----------------------------------------------------------------------------
print("🔍 DIAGNÓSTICO DEL CAMPO productos_ean")
print("=" * 60)

# ¿Qué valores tiene?
print("\nTop 10 valores de productos_ean:")
print(df_productos["productos_ean"].value_counts(dropna=False).head(10))

# ¿Cuántos nulos?
n_null = df_productos["productos_ean"].isna().sum()
print(f"\nNulos en productos_ean: {n_null:,}")

# Veamos algunas filas crudas para entender qué pasó
print("\nPrimeras 5 filas completas (columnas clave):")
cols_clave = ["id_comercio", "id_bandera", "id_producto", "productos_ean",
              "productos_descripcion", "productos_precio_lista"]
print(df_productos[cols_clave].head().to_string())

🔍 DIAGNÓSTICO DEL CAMPO productos_ean

Top 10 valores de productos_ean:
productos_ean
1    15049145
0      175227
Name: count, dtype: Int64

Nulos en productos_ean: 0

Primeras 5 filas completas (columnas clave):
   id_comercio  id_bandera    id_producto productos_ean productos_descripcion  productos_precio_lista
0           15           1  0000002674407             1         PRESERVATIVOS                  5486.0
1           15           1  0000040084107             1    HUEVO DE CHOCOLATE                  2784.0
2           15           1  0000042277057             1  CREM FACIAL CUID NUT                  6150.0
3           15           1  0000075079437             1  ANTITRANS CREM WOMEN                 16999.0
4           15           1  0000075079444             1   ANTITRANS CREMA MEN                 16999.0


In [9]:
# -----------------------------------------------------------------------------
# 2.5.B) Optimización de memoria de df_productos
# -----------------------------------------------------------------------------
# Estrategia:
#   1) Columnas de texto muy repetitivas → 'category' (ahorra 80-95% memoria).
#   2) IDs Int64 → int32 donde el rango lo permita.
#   3) Floats → float32 (precisión suficiente para precios).
#   4) Descartamos columnas que no vamos a usar todavía (promociones).
# =============================================================================

print("\n🧹 OPTIMIZACIÓN DE MEMORIA")
print("=" * 60)

mem_antes = df_productos.memory_usage(deep=True).sum() / (1024**2)
print(f"Memoria antes: {mem_antes:,.1f} MB")

# 1) Convertir a category: columnas con pocas categorías únicas relativas
#    al total de filas. Regla práctica: si #únicos / #filas < 0.5, conviene.
cols_a_categoria = [
    "productos_unidad_medida_presentacion",
    "productos_marca",
    "productos_unidad_medida_referencia",
    "productos_leyenda_promo1",
    "productos_leyenda_promo2",
    "cadena_folder",
]
for col in cols_a_categoria:
    if col in df_productos.columns:
        df_productos[col] = df_productos[col].astype("category")

# 2) IDs: de Int64 (8 bytes + bitmask) a int32 (4 bytes). Antes chequeamos
#    que los valores entren en int32 (max ~2.1 billones, sobra).
for col in ["id_comercio", "id_bandera", "id_sucursal"]:
    if col in df_productos.columns:
        # Rellenamos nulos con -1 para poder pasar a int32 (int32 no admite NaN)
        df_productos[col] = df_productos[col].fillna(-1).astype("int32")

# 3) Floats: float64 → float32 (precisión de ~7 dígitos, más que suficiente
#    para precios en pesos argentinos).
cols_float = [
    "productos_cantidad_presentacion",
    "productos_precio_lista",
    "productos_precio_referencia",
    "productos_cantidad_referencia",
    "productos_precio_unitario_promo1",
    "productos_precio_unitario_promo2",
]
for col in cols_float:
    if col in df_productos.columns:
        df_productos[col] = df_productos[col].astype("float32")

mem_despues = df_productos.memory_usage(deep=True).sum() / (1024**2)
ahorro_pct = (1 - mem_despues/mem_antes) * 100
print(f"Memoria después: {mem_despues:,.1f} MB")
print(f"Ahorro: {ahorro_pct:.1f}%")

# Mostramos el desglose por columna para ver dónde está el peso
print("\nMemoria por columna (top 10):")
mem_por_col = (df_productos.memory_usage(deep=True) / (1024**2))
print(mem_por_col.sort_values(ascending=False).head(10).round(1))


🧹 OPTIMIZACIÓN DE MEMORIA
Memoria antes: 9,348.6 MB
Memoria después: 3,548.3 MB
Ahorro: 62.0%

Memoria por columna (top 10):
productos_descripcion                                                                                                            1151.3
id_producto                                                                                                                       900.2
productos_ean                                                                                                                     726.0
productos_leyenda_promo2                                                                                                          116.2
productos_precio_unitario_promo2                                                                                                   58.1
id_sucursal                                                                                                                        58.1
id_bandera                                                

In [10]:
# =============================================================================
# PASO 2.6: Validación del EAN y optimización agresiva de memoria
# =============================================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 2.6.A) Validación rigurosa: ¿id_producto es realmente un EAN-13?
# -----------------------------------------------------------------------------
print("🔍 VALIDACIÓN DE id_producto COMO EAN-13")
print("=" * 60)

# Distribución de longitudes del id_producto
longitudes = df_productos["id_producto"].str.len().value_counts().sort_index()
print("\nDistribución de longitudes de id_producto:")
print(longitudes)

# ¿Cuántos id_producto son numéricos puros?
es_numerico = df_productos["id_producto"].str.match(r"^\d+$", na=False)
print(f"\nid_producto numéricos puros: {es_numerico.sum():>12,} "
      f"({100*es_numerico.mean():.2f}%)")

# Cross-tab: flag productos_ean vs longitud del id_producto
# Esto nos confirma si el flag está bien puesto
print("\nCross-tab: productos_ean (flag) vs longitud de id_producto:")
ct = pd.crosstab(
    df_productos["productos_ean"],
    df_productos["id_producto"].str.len().fillna(-1).astype(int),
    margins=True,
)
# Mostramos solo las longitudes más frecuentes (sino es un quilombo)
cols_top = longitudes.head(8).index.tolist() + ["All"]
print(ct[[c for c in cols_top if c in ct.columns]])

# ¿Cuántos id_producto ÚNICOS hay? (este sí es el número interesante)
n_id_producto_unicos = df_productos["id_producto"].nunique()
print(f"\n✨ id_producto únicos en toda la base: {n_id_producto_unicos:,}")
print("   (este es el número real de 'productos distintos' candidatos a canasta)")

# Cuántos id_producto aparecen en múltiples cadenas (los "comparables")
productos_por_razon_social = (
    df_productos.merge(
        df_comercio[["id_comercio", "id_bandera", "comercio_razon_social"]],
        on=["id_comercio", "id_bandera"],
        how="left",
    )
    .groupby("id_producto")["comercio_razon_social"]
    .nunique()
)
print(f"\nid_producto que aparecen en ≥2 razones sociales distintas: "
      f"{(productos_por_razon_social >= 2).sum():,}")
print(f"id_producto que aparecen en ≥5 razones sociales distintas: "
      f"{(productos_por_razon_social >= 5).sum():,}")

🔍 VALIDACIÓN DE id_producto COMO EAN-13

Distribución de longitudes de id_producto:
id_producto
13    15224372
Name: count, dtype: Int64

id_producto numéricos puros:   15,224,372 (100.00%)

Cross-tab: productos_ean (flag) vs longitud de id_producto:
id_producto          13       All
productos_ean                    
0                175227    175227
1              15049145  15049145
All            15224372  15224372

✨ id_producto únicos en toda la base: 87,613
   (este es el número real de 'productos distintos' candidatos a canasta)

id_producto que aparecen en ≥2 razones sociales distintas: 22,186
id_producto que aparecen en ≥5 razones sociales distintas: 7,250


In [11]:
# -----------------------------------------------------------------------------
# 2.6.B) Optimización agresiva de memoria
# -----------------------------------------------------------------------------
# Tres cambios:
#   1) productos_ean → bool (era 15M de "0"/"1", ahora ocupa 1 byte/fila).
#   2) id_producto → category (pocos únicos relativos al total de filas).
#   3) productos_descripcion → category (idem, muchas descripciones se repiten
#      porque el mismo producto se vende en miles de sucursales).
#   4) Descartamos columnas que no vamos a usar en el corto plazo.
# =============================================================================

print("\n🧹 OPTIMIZACIÓN AGRESIVA DE MEMORIA")
print("=" * 60)

mem_antes = df_productos.memory_usage(deep=True).sum() / (1024**2)
print(f"Memoria antes: {mem_antes:,.1f} MB")

# 1) productos_ean como booleano (flag "tiene EAN real")
#    Convertimos: "1" → True, "0" → False
df_productos["tiene_ean"] = (df_productos["productos_ean"] == 1)
df_productos = df_productos.drop(columns=["productos_ean"])

# 2) id_producto a category (la clave para matchear entre cadenas)
df_productos["id_producto"] = df_productos["id_producto"].astype("category")

# 3) productos_descripcion a category
#    Esta es la conversión que más memoria ahorra: el mismo texto
#    "Botella coca cola PVC-1.5-lt." se repite en miles de sucursales.
df_productos["productos_descripcion"] = (
    df_productos["productos_descripcion"].astype("category")
)

# 4) Descartamos columnas de promos que no vamos a usar ahora
#    (las podemos recuperar después releyendo si hace falta)
cols_a_dropear = [
    "productos_precio_unitario_promo1",
    "productos_leyenda_promo1",
    "productos_precio_unitario_promo2",
    "productos_leyenda_promo2",
]
df_productos = df_productos.drop(columns=cols_a_dropear, errors="ignore")

mem_despues = df_productos.memory_usage(deep=True).sum() / (1024**2)
ahorro_pct = (1 - mem_despues/mem_antes) * 100
print(f"Memoria después: {mem_despues:,.1f} MB")
print(f"Ahorro adicional: {ahorro_pct:.1f}%")
print(f"Ahorro acumulado desde el inicio (9348 MB): "
      f"{100*(1 - mem_despues/9348.6):.1f}%")

print("\nMemoria por columna (tras la optimización):")
print((df_productos.memory_usage(deep=True) / (1024**2))
      .sort_values(ascending=False).round(1))

print("\nDtypes finales de df_productos:")
print(df_productos.dtypes)


🧹 OPTIMIZACIÓN AGRESIVA DE MEMORIA
Memoria antes: 3,548.3 MB
Memoria después: 763.8 MB
Ahorro adicional: 78.5%
Ahorro acumulado desde el inicio (9348 MB): 91.8%

Memoria por columna (tras la optimización):
productos_leyenda_promo2                                                                                                         116.2
productos_descripcion                                                                                                             73.5
id_producto                                                                                                                       65.3
id_comercio                                                                                                                       58.1
productos_cantidad_presentacion                                                                                                   58.1
id_sucursal                                                                                                           

In [12]:
# =============================================================================
# PASO 3: Exploración estructural de la base
# =============================================================================
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 3.0) Limpieza fina: dropear columnas residuales de promos si quedaron
# -----------------------------------------------------------------------------
cols_promo_residuales = [c for c in df_productos.columns
                          if "promo" in c.lower() or "leyenda" in c.lower()]
if cols_promo_residuales:
    print(f"🧹 Dropeo columnas residuales: {cols_promo_residuales}")
    df_productos = df_productos.drop(columns=cols_promo_residuales)
    mem = df_productos.memory_usage(deep=True).sum() / (1024**2)
    print(f"   Memoria actual: {mem:.1f} MB\n")

# -----------------------------------------------------------------------------
# 3.1) ¿Qué es una "cadena"? Entendiendo comercio vs bandera
# -----------------------------------------------------------------------------
# IMPORTANTE: en SEPA, un "comercio" (razón social) puede operar bajo varias
# "banderas" (marcas comerciales). Ejemplo clásico:
#   - Cencosud S.A. (1 razón social) → Vea, Disco, Jumbo (3 banderas)
#   - INC S.A.      (1 razón social) → Carrefour, Express, Market, Maxi...
# Para análisis de precios, lo que importa es la BANDERA (el consumidor ve
# "Jumbo", no "Cencosud S.A."), así que vamos a trabajar a ese nivel.
# =============================================================================

print("🏢 ESTRUCTURA DE COMERCIOS Y BANDERAS")
print("=" * 70)
print(f"\nTabla de comercios (razón social → bandera):")
# Mostramos la relación razón social → bandera, con su cuit
vista_comercio = (df_comercio[["comercio_razon_social",
                                "comercio_bandera_nombre",
                                "comercio_cuit",
                                "id_comercio", "id_bandera"]]
                   .drop_duplicates()
                   .sort_values(["comercio_razon_social",
                                 "comercio_bandera_nombre"])
                   .reset_index(drop=True))
print(vista_comercio.to_string(index=False))

# -----------------------------------------------------------------------------
# 3.2) Volumen de datos por bandera: ¿quién reporta cuánto?
# -----------------------------------------------------------------------------
# Esto nos dice qué banderas tienen presencia real en la muestra. Las que
# reportan 50 sucursales y 1 millón de productos son las "grandes"; las que
# reportan 2 sucursales y 300 productos son comercios de barrio.
# =============================================================================
print("\n\n📊 VOLUMEN POR BANDERA")
print("=" * 70)

# Merge ligero de df_productos con comercio para tener el nombre de bandera
# Usamos groupby directo para no materializar el merge completo (ahorra RAM)
vol_por_bandera = (df_productos
    .groupby(["id_comercio", "id_bandera"], observed=True)
    .agg(n_filas=("id_producto", "size"),
         n_sucursales=("id_sucursal", "nunique"),
         n_productos=("id_producto", "nunique"))
    .reset_index()
    .merge(df_comercio[["id_comercio", "id_bandera",
                        "comercio_razon_social",
                        "comercio_bandera_nombre"]].drop_duplicates(),
           on=["id_comercio", "id_bandera"], how="left")
    .sort_values("n_filas", ascending=False)
    .reset_index(drop=True))

# Mostramos una vista legible
print(vol_por_bandera[["comercio_razon_social", "comercio_bandera_nombre",
                        "n_sucursales", "n_productos", "n_filas"]]
      .to_string(index=False))

# -----------------------------------------------------------------------------
# 3.3) Distribución geográfica de sucursales
# -----------------------------------------------------------------------------
# Queremos saber: ¿en qué provincias tenemos cobertura para comparar?
# Los códigos que vemos (AR-C, AR-B, etc.) son ISO 3166-2:
#   AR-C = CABA, AR-B = Buenos Aires, AR-X = Córdoba, AR-S = Santa Fe, etc.
# =============================================================================
print("\n\n🗺️  DISTRIBUCIÓN GEOGRÁFICA")
print("=" * 70)

# Diccionario ISO 3166-2 para Argentina (provincias)
ISO_A_PROVINCIA = {
    "AR-C": "CABA",
    "AR-B": "Buenos Aires",
    "AR-K": "Catamarca",
    "AR-H": "Chaco",
    "AR-U": "Chubut",
    "AR-X": "Córdoba",
    "AR-W": "Corrientes",
    "AR-E": "Entre Ríos",
    "AR-P": "Formosa",
    "AR-Y": "Jujuy",
    "AR-L": "La Pampa",
    "AR-F": "La Rioja",
    "AR-M": "Mendoza",
    "AR-N": "Misiones",
    "AR-Q": "Neuquén",
    "AR-R": "Río Negro",
    "AR-A": "Salta",
    "AR-J": "San Juan",
    "AR-D": "San Luis",
    "AR-Z": "Santa Cruz",
    "AR-S": "Santa Fe",
    "AR-G": "Santiago del Estero",
    "AR-V": "Tierra del Fuego",
    "AR-T": "Tucumán",
}

df_sucursales["provincia_nombre"] = (
    df_sucursales["sucursales_provincia"].map(ISO_A_PROVINCIA)
    .fillna(df_sucursales["sucursales_provincia"])
)

# Sucursales por provincia (todas)
sucursales_por_prov = (df_sucursales
    .groupby("provincia_nombre")
    .agg(n_sucursales=("id_sucursal", "count"),
         n_banderas=("id_bandera", "nunique"),
         n_comercios=("id_comercio", "nunique"))
    .sort_values("n_sucursales", ascending=False))

print("\nSucursales por provincia:")
print(sucursales_por_prov.to_string())

# -----------------------------------------------------------------------------
# 3.4) Productos candidatos a canasta: los más "comparables"
# -----------------------------------------------------------------------------
# Criterio: un buen producto para la canasta es el que:
#   a) Aparece en MUCHAS banderas distintas (comparabilidad).
#   b) Aparece en MUCHAS sucursales (robustez de la muestra).
#   c) Tiene precio > 0 (datos válidos).
# =============================================================================
print("\n\n🛒 PRODUCTOS CANDIDATOS A CANASTA")
print("=" * 70)

# Filtramos precios válidos para este análisis
mascara_precio_valido = (df_productos["productos_precio_lista"] > 0)
print(f"\nFilas con precio válido: {mascara_precio_valido.sum():,} "
      f"de {len(df_productos):,} ({100*mascara_precio_valido.mean():.2f}%)")

# Para cada id_producto, cuántas banderas distintas lo venden
# (usamos la combinación id_comercio + id_bandera como "bandera única")
prods_candidatos = (df_productos[mascara_precio_valido]
    .assign(bandera_id=lambda d: d["id_comercio"].astype(str)
                                 + "_" + d["id_bandera"].astype(str))
    .groupby("id_producto", observed=True)
    .agg(n_banderas=("bandera_id", "nunique"),
         n_sucursales=("id_sucursal", "nunique"),
         precio_mediano=("productos_precio_lista", "median"),
         descripcion=("productos_descripcion", "first"))
    .sort_values("n_banderas", ascending=False))

print(f"\nTop 20 productos por cobertura (n_banderas):")
print(prods_candidatos.head(20).to_string())

# Cuántos productos cumplen distintos umbrales
print(f"\nDistribución de cobertura (productos × n_banderas):")
for umbral in [2, 3, 5, 10, 15, 20]:
    n = (prods_candidatos["n_banderas"] >= umbral).sum()
    print(f"   ≥ {umbral:2d} banderas: {n:>6,} productos")

# Guardamos esta tabla para el próximo paso
print(f"\n✅ Paso 3 completo. Nuevas variables:")
print(f"   - vista_comercio        ({len(vista_comercio)} filas)")
print(f"   - vol_por_bandera       ({len(vol_por_bandera)} filas)")
print(f"   - sucursales_por_prov   ({len(sucursales_por_prov)} filas)")
print(f"   - prods_candidatos      ({len(prods_candidatos):,} filas)")

🧹 Dropeo columnas residuales: ['productos_leyenda_promo2                                                                                                     ']
   Memoria actual: 647.7 MB

🏢 ESTRUCTURA DE COMERCIOS Y BANDERAS

Tabla de comercios (razón social → bandera):
                                                 comercio_razon_social                           comercio_bandera_nombre comercio_cuit  id_comercio  id_bandera
                                                         Agapantus sa.                                          SuperCLC   30711661871           36           6
                                                          Alberdi S.A.                                      Maxi Comodin   30578411174            6           2
                                                          Alberdi S.A.                             Supermercados Comodin   30578411174            6           1
                         COTO CENTRO INTEGRAL DE COMERCIALIZACION S.A.                  

In [13]:
# =============================================================================
# PASO 4: Filtros, enriquecimiento y base analítica
# =============================================================================
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 4.1) Filtrar farmacias y estaciones de servicio
# -----------------------------------------------------------------------------
# Lista explícita de banderas a excluir. La hacemos por nombre de bandera
# (no por razón social) porque ya dijimos que trabajamos a nivel bandera.
# =============================================================================

BANDERAS_EXCLUIDAS = {
    "FARMACITY",        # Farmacia
    "SIMPLICITY",       # Farmacia (Farmcity S.A.)
    "FULL",             # Estación de servicio YPF
    "Axion Energy",     # Estación de servicio
    "ESTACION LIMA S.A.",  # Estación de servicio (310 filas, lo sacamos)
}

print("🔪 FILTRADO DE BANDERAS NO-SUPERMERCADO")
print("=" * 60)

# Identificamos los (id_comercio, id_bandera) a excluir
banderas_excluidas_ids = (df_comercio
    .loc[df_comercio["comercio_bandera_nombre"].isin(BANDERAS_EXCLUIDAS),
         ["id_comercio", "id_bandera", "comercio_bandera_nombre"]]
    .drop_duplicates())
print("\nBanderas a excluir:")
print(banderas_excluidas_ids.to_string(index=False))

# Creamos un set de tuplas (id_comercio, id_bandera) para filtrar rápido
tuplas_excluir = set(zip(banderas_excluidas_ids["id_comercio"],
                          banderas_excluidas_ids["id_bandera"]))

# Aplicamos el filtro a los tres DataFrames
def filtrar_banderas(df, tuplas):
    """Filtra un DataFrame quitando las filas de las banderas excluidas."""
    mascara = ~df[["id_comercio", "id_bandera"]].apply(tuple, axis=1).isin(tuplas)
    return df[mascara].copy()

n_antes_prod = len(df_productos)
n_antes_suc  = len(df_sucursales)
n_antes_com  = len(df_comercio)

df_productos  = filtrar_banderas(df_productos,  tuplas_excluir)
df_sucursales = filtrar_banderas(df_sucursales, tuplas_excluir)
df_comercio   = filtrar_banderas(df_comercio,   tuplas_excluir)

print(f"\nFilas removidas:")
print(f"   df_productos:  {n_antes_prod:>12,} → {len(df_productos):>12,} "
      f"({n_antes_prod - len(df_productos):,} menos)")
print(f"   df_sucursales: {n_antes_suc:>12,} → {len(df_sucursales):>12,}")
print(f"   df_comercio:   {n_antes_com:>12,} → {len(df_comercio):>12,}")

# -----------------------------------------------------------------------------
# 4.2) Filtros adicionales de calidad sobre productos
# -----------------------------------------------------------------------------
# Quitamos filas con precio_lista <= 0 o NaN. Estos son errores de carga
# que no aportan información y pueden romper promedios.
# =============================================================================

print("\n\n🧽 FILTROS DE CALIDAD DE PRECIO")
print("=" * 60)

n_antes = len(df_productos)
mascara_precio_ok = (df_productos["productos_precio_lista"].notna() &
                      (df_productos["productos_precio_lista"] > 0))
df_productos = df_productos[mascara_precio_ok].copy()
print(f"\nFilas con precio inválido removidas: "
      f"{n_antes - len(df_productos):,}")
print(f"df_productos final: {len(df_productos):,} filas")

# -----------------------------------------------------------------------------
# 4.3) Enriquecer df_sucursales con el nombre de bandera y provincia
# -----------------------------------------------------------------------------
# Esto nos va a permitir agregar por bandera o provincia sin hacer merges
# pesados con df_productos más adelante.
# =============================================================================

# Mapa provincia ISO → nombre (ya lo teníamos en el paso anterior)
ISO_A_PROVINCIA = {
    "AR-C": "CABA", "AR-B": "Buenos Aires", "AR-K": "Catamarca",
    "AR-H": "Chaco", "AR-U": "Chubut", "AR-X": "Córdoba",
    "AR-W": "Corrientes", "AR-E": "Entre Ríos", "AR-P": "Formosa",
    "AR-Y": "Jujuy", "AR-L": "La Pampa", "AR-F": "La Rioja",
    "AR-M": "Mendoza", "AR-N": "Misiones", "AR-Q": "Neuquén",
    "AR-R": "Río Negro", "AR-A": "Salta", "AR-J": "San Juan",
    "AR-D": "San Luis", "AR-Z": "Santa Cruz", "AR-S": "Santa Fe",
    "AR-G": "Santiago del Estero", "AR-V": "Tierra del Fuego",
    "AR-T": "Tucumán",
}

df_sucursales["provincia"] = (df_sucursales["sucursales_provincia"]
                               .map(ISO_A_PROVINCIA)
                               .fillna(df_sucursales["sucursales_provincia"]))

# Agregamos el nombre de la bandera haciendo merge con df_comercio
df_sucursales = df_sucursales.merge(
    df_comercio[["id_comercio", "id_bandera",
                 "comercio_razon_social", "comercio_bandera_nombre"]]
    .drop_duplicates(),
    on=["id_comercio", "id_bandera"],
    how="left",
)

print("\n\n🔗 df_sucursales enriquecido")
print("=" * 60)
print(f"Columnas clave: id_sucursal, id_comercio, id_bandera, "
      f"comercio_bandera_nombre, provincia, sucursales_localidad, "
      f"lat/lon")
print(f"Total sucursales activas: {len(df_sucursales):,}")

# -----------------------------------------------------------------------------
# 4.4) Tabla maestra: df_precios con precio, bandera y provincia en una fila
# -----------------------------------------------------------------------------
# Esta es LA tabla que vamos a usar para análisis. Un merge controlado
# entre productos y sucursales para tener todo junto. Solo traemos las
# columnas estrictamente necesarias — df_productos tiene millones de filas,
# queremos mantener la memoria bajo control.
# =============================================================================

print("\n\n🧱 CONSTRUCCIÓN DE LA TABLA MAESTRA df_precios")
print("=" * 60)

# Columnas mínimas de sucursales que necesitamos
cols_suc_min = ["id_comercio", "id_bandera", "id_sucursal",
                "comercio_bandera_nombre", "provincia",
                "sucursales_localidad", "sucursales_latitud",
                "sucursales_longitud"]

# Merge
df_precios = df_productos.merge(
    df_sucursales[cols_suc_min],
    on=["id_comercio", "id_bandera", "id_sucursal"],
    how="inner",   # inner: si una sucursal no está en df_sucursales, tiramos
)

# Achicamos un poco: convertimos columnas de texto nuevas a category
df_precios["comercio_bandera_nombre"] = df_precios["comercio_bandera_nombre"].astype("category")
df_precios["provincia"]                = df_precios["provincia"].astype("category")
df_precios["sucursales_localidad"]     = df_precios["sucursales_localidad"].astype("category")

mem = df_precios.memory_usage(deep=True).sum() / (1024**2)
print(f"\ndf_precios: {len(df_precios):,} filas, {mem:,.1f} MB")
print(f"\nColumnas:")
print(df_precios.dtypes)

# Chequeo: ¿perdimos filas en el merge? (producto sin sucursal asociada)
n_perdidas = len(df_productos) - len(df_precios)
print(f"\nFilas sin match con sucursal (descartadas): {n_perdidas:,} "
      f"({100*n_perdidas/len(df_productos):.3f}%)")

# -----------------------------------------------------------------------------
# 4.5) Resumen final de la base analítica
# -----------------------------------------------------------------------------
print("\n\n📊 RESUMEN DE LA BASE ANALÍTICA")
print("=" * 60)

print(f"\n🏬 Banderas activas:     {df_precios['comercio_bandera_nombre'].nunique()}")
print(f"🏪 Sucursales activas:   {df_precios['id_sucursal'].nunique():,}")
print(f"🛒 Productos únicos:     {df_precios['id_producto'].nunique():,}")
print(f"🗺️  Provincias con datos: {df_precios['provincia'].nunique()}")
print(f"📏 Filas totales:        {len(df_precios):,}")

print("\nFilas por bandera:")
print(df_precios["comercio_bandera_nombre"]
      .value_counts()
      .to_string())

print("\nSucursales por bandera × provincia (para detectar huecos):")
tabla_cobertura = (df_sucursales
    .groupby(["comercio_bandera_nombre", "provincia"], observed=True)
    .size()
    .unstack(fill_value=0))
print(tabla_cobertura.to_string())

print(f"\n✅ Paso 4 completo. Variable clave: df_precios")

🔪 FILTRADO DE BANDERAS NO-SUPERMERCADO

Banderas a excluir:
 id_comercio  id_bandera comercio_bandera_nombre
          24           1              SIMPLICITY
          24           2               FARMACITY
          19           1                    FULL
          23           1            Axion Energy
           4           1      ESTACION LIMA S.A.

Filas removidas:
   df_productos:    15,224,372 →   14,808,038 (416,334 menos)
   df_sucursales:        3,168 →        2,629
   df_comercio:             37 →           32


🧽 FILTROS DE CALIDAD DE PRECIO

Filas con precio inválido removidas: 147
df_productos final: 14,807,891 filas


🔗 df_sucursales enriquecido
Columnas clave: id_sucursal, id_comercio, id_bandera, comercio_bandera_nombre, provincia, sucursales_localidad, lat/lon
Total sucursales activas: 2,629


🧱 CONSTRUCCIÓN DE LA TABLA MAESTRA df_precios

df_precios: 14,728,809 filas, 908.3 MB

Columnas:
id_comercio                                int32
id_bandera                      

In [15]:
# =============================================================================
# PASO 5: Definición de la canasta de consumo
# =============================================================================
# Estrategia:
#   1) Calcular métricas por producto (cobertura, variabilidad, precio típico).
#   2) Aplicar filtros duros de calidad (cobertura mínima, varianza tolerable).
#   3) Clasificar productos en rubros usando keywords sobre la descripción.
#   4) Dejar todo listo para que elijas la canasta final.
# =============================================================================
import pandas as pd
import numpy as np
import re
import unicodedata

# -----------------------------------------------------------------------------
# 5.1) Métricas por producto (id_producto)
# -----------------------------------------------------------------------------
# Para cada producto calculamos:
#   - n_banderas:        en cuántas banderas se vende (cobertura horizontal).
#   - n_sucursales:      en cuántas sucursales se registra (masividad).
#   - n_provincias:      en cuántas provincias (cobertura geográfica).
#   - precio_mediano:    precio típico (mediana es robusta a outliers).
#   - precio_cv:         coef. de variación (σ/μ). Alto → precio errático.
#   - precio_p10/p90:    percentiles 10 y 90 (rango típico sin outliers).
#   - descripcion:       la descripción más frecuente (moda).
#   - marca:             la marca más frecuente.
#   - cant_presentacion: cantidad típica (ej: 1.5 lt).
#   - unidad_presentacion: unidad típica (lt, kg, un).
# =============================================================================

print("🔬 CALCULANDO MÉTRICAS POR PRODUCTO")
print("=" * 60)

# Pre-trabajo: "bandera_id" como identificador único de bandera
# (combinamos id_comercio + id_bandera porque id_bandera solo no es único
# entre comercios distintos)
df_precios["bandera_id"] = (df_precios["id_comercio"].astype(str)
                             + "_" + df_precios["id_bandera"].astype(str))

# Función helper: la moda (valor más frecuente). Si hay empate, devuelve el primero.
def moda_segura(serie):
    modo = serie.mode()
    return modo.iloc[0] if len(modo) > 0 else np.nan

# Agregamos todas las métricas en un solo groupby
# IMPORTANTE: observed=True en groupby con categorías para no generar
# todas las combinaciones posibles (ahorra RAM y tiempo)
print("⏳ Agregando... (puede tardar 1-2 min por el volumen)")

productos_metricas = (df_precios
    .groupby("id_producto", observed=True)
    .agg(
        n_banderas=("bandera_id", "nunique"),
        n_sucursales=("id_sucursal", "nunique"),
        n_provincias=("provincia", "nunique"),
        n_obs=("productos_precio_lista", "size"),
        precio_mediano=("productos_precio_lista", "median"),
        precio_media=("productos_precio_lista", "mean"),
        precio_std=("productos_precio_lista", "std"),
        precio_p10=("productos_precio_lista",
                     lambda s: s.quantile(0.10)),
        precio_p90=("productos_precio_lista",
                     lambda s: s.quantile(0.90)),
        descripcion=("productos_descripcion", moda_segura),
        marca=("productos_marca", moda_segura),
        cant_presentacion=("productos_cantidad_presentacion", moda_segura),
        unidad_presentacion=("productos_unidad_medida_presentacion", moda_segura),
    )
    .reset_index()
)

# Calculamos el coef. de variación: σ/μ. Valor típico entre 0 y 1.
# CV > 0.5 suele indicar precios inconsistentes (error de carga, cambio de formato).
productos_metricas["precio_cv"] = (productos_metricas["precio_std"]
                                    / productos_metricas["precio_media"])

# Ratio p90/p10: otra medida de dispersión, más robusta que el CV
productos_metricas["ratio_p90_p10"] = (productos_metricas["precio_p90"]
                                        / productos_metricas["precio_p10"])

print(f"✅ Tabla productos_metricas construida: {len(productos_metricas):,} productos\n")

# -----------------------------------------------------------------------------
# 5.2) Diagnóstico: distribución de las métricas
# -----------------------------------------------------------------------------
print("📊 DISTRIBUCIÓN DE MÉTRICAS DE COBERTURA Y CALIDAD")
print("=" * 60)

print("\nCobertura (n_banderas):")
print(productos_metricas["n_banderas"].describe().to_string())

print("\nCobertura geográfica (n_provincias):")
print(productos_metricas["n_provincias"].describe().to_string())

print("\nConsistencia de precio (ratio_p90_p10):")
print(productos_metricas["ratio_p90_p10"].describe().to_string())

# -----------------------------------------------------------------------------
# 5.3) Filtros duros: definición del universo "canastable"
# -----------------------------------------------------------------------------
# Criterios:
#   - n_banderas >= 8:     presente en al menos 8 banderas de supermercados.
#                          Con 22 banderas activas, esto es >35% de cobertura.
#   - n_provincias >= 5:   presente en al menos 5 provincias.
#   - ratio_p90_p10 <= 3:  precios razonablemente consistentes (entre extremos,
#                          el más caro no es más de 3x el más barato).
#                          Esto descarta productos con errores de carga graves
#                          tipo "Coca 2L a $0.01" o confusiones de formato.
#   - precio_mediano > 50: descarta precios ínfimos que son errores (ya vimos
#                          "Coca a $0.31" en tu muestra inicial).
# =============================================================================

FILTRO_BANDERAS_MIN   = 8
FILTRO_PROVINCIAS_MIN = 5
FILTRO_RATIO_MAX      = 3.0
FILTRO_PRECIO_MIN     = 50.0

mask_canastables = (
    (productos_metricas["n_banderas"]    >= FILTRO_BANDERAS_MIN) &
    (productos_metricas["n_provincias"]  >= FILTRO_PROVINCIAS_MIN) &
    (productos_metricas["ratio_p90_p10"] <= FILTRO_RATIO_MAX) &
    (productos_metricas["precio_mediano"] >= FILTRO_PRECIO_MIN)
)

canastables = productos_metricas[mask_canastables].copy()
print(f"\n🎯 FILTROS APLICADOS")
print("=" * 60)
print(f"   n_banderas    ≥ {FILTRO_BANDERAS_MIN}")
print(f"   n_provincias  ≥ {FILTRO_PROVINCIAS_MIN}")
print(f"   ratio p90/p10 ≤ {FILTRO_RATIO_MAX}")
print(f"   precio_mediano ≥ ${FILTRO_PRECIO_MIN}")
print(f"\n✨ Productos canastables: {len(canastables):,} de "
      f"{len(productos_metricas):,} ({100*len(canastables)/len(productos_metricas):.1f}%)")

# -----------------------------------------------------------------------------
# 5.4) Clasificación en rubros usando keywords sobre la descripción
# -----------------------------------------------------------------------------
# Normalizamos la descripción (minúsculas, sin acentos) y aplicamos un
# árbol de reglas por keywords. Orden importa: evaluamos primero lo más
# específico, después lo general.
# =============================================================================

def normalizar_texto(s):
    """Minúsculas y sin acentos, para matchear con keywords."""
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s

canastables["desc_norm"] = canastables["descripcion"].apply(normalizar_texto)

# Reglas de clasificación: (rubro, lista_de_patrones_regex)
# Los patrones son regex; \b = límite de palabra
REGLAS_RUBRO = [
    # Bebidas sin alcohol
    ("Bebidas", [
        r"\bgaseos", r"\bcola\b", r"\bsprite", r"\bfanta", r"\bpepsi",
        r"\bmanaos", r"\bmirinda", r"\b7[ -]?up\b",
        r"\bagua\b", r"\bmineral", r"\bsin gas", r"\bcon gas",
        r"\bisot", r"\bpowerade", r"\bgatorade",
        r"\bjugo\b", r"\bnectar", r"\bexprimido",
        r"\bmate coc", r"\byerba\b", r"\bte\b", r"\bcafe\b", r"\bcafet",
        r"\bchocolatada", r"\bleche choc",
    ]),
    # Alcohol
    ("Alcohol", [
        r"\bvino\b", r"\bcerveza", r"\bfernet", r"\bwhisky", r"\bwhiskey",
        r"\bvodka", r"\bgin\b", r"\bron\b", r"\btequila", r"\baperitivo",
        r"\bespumante", r"\bchampag",
    ]),
    # Lácteos
    ("Lácteos", [
        r"\bleche\b(?! choc)", r"\byogu", r"\bmanteca", r"\bqueso",
        r"\bricota", r"\bcrema\b", r"\bdulce de leche", r"\bdulce leche",
        r"\bdanon", r"\bserenito",
    ]),
    # Carnes y fiambres
    ("Carnes y fiambres", [
        r"\bcarne\b", r"\bpollo\b", r"\bcerdo\b", r"\bmilanesa",
        r"\bhamburg", r"\bsalchich", r"\bchorizo", r"\bmorcilla",
        r"\bjamon\b", r"\bsalame", r"\bsalami", r"\bmortadela",
        r"\bbondiola", r"\bpanceta", r"\bfiambre",
    ]),
    # Panadería y harinas
    ("Panadería y harinas", [
        r"\bpan\b", r"\btostada", r"\bgalle", r"\bgaletit",
        r"\bbizcoch", r"\bfacturas", r"\bmedialunas",
        r"\bharina", r"\bfideo", r"\bpasta\b", r"\bpolenta",
        r"\barroz", r"\bavena",
    ]),
    # Almacén / cocina
    ("Almacén", [
        r"\baceite", r"\bvinagre", r"\bsal\b", r"\bazucar",
        r"\bmayones", r"\bketchup", r"\bmostaza", r"\bsalsa",
        r"\btomate\b", r"\bpure\b", r"\bpuré",
        r"\blegumbre", r"\blenteja", r"\bporoto", r"\bgarbanzo",
        r"\bconserv", r"\batun\b", r"\bsardina", r"\bcaballa",
        r"\bmermelad", r"\bdulce\b",
    ]),
    # Snacks y golosinas
    ("Snacks y golosinas", [
        r"\balfajor", r"\bchocolate", r"\bbombon", r"\bcaram",
        r"\bchupetin", r"\bgoma\b", r"\bturron",
        r"\bpapas frit", r"\bpalit", r"\bsnack", r"\bchizit", r"\bnachos",
    ]),
    # Limpieza hogar
    ("Limpieza", [
        r"\blavandina", r"\bdetergente", r"\blavavajill",
        r"\bjabon\b.*polvo", r"\bjabon\b.*liqu",
        r"\bsuavizante", r"\bperfumante", r"\bdesinfect",
        r"\blimpiador", r"\blimpia\b", r"\bmultiuso",
        r"\bpapel higi", r"\bservilleta", r"\brollo cocina",
    ]),
    # Higiene personal
    ("Higiene personal", [
        r"\bshampoo", r"\bacondicion", r"\bjabon\b(?!.*polvo)(?!.*liqu)",
        r"\bdesodor", r"\bantitrans", r"\bcrema\b.*corp",
        r"\bpasta dental", r"\bcepillo dent", r"\bdental",
        r"\btoalla\b", r"\btampon", r"\bpanal", r"\bpañal",
        r"\bafeit", r"\bgillette", r"\brasurad",
    ]),
    # Bebé
    ("Bebé", [
        r"\bpanal\b", r"\bpañal", r"\bleche.*bebe", r"\bnan\b",
        r"\bhuggies", r"\bpampers",
    ]),
]

def clasificar_rubro(desc_norm):
    for rubro, patrones in REGLAS_RUBRO:
        for patron in patrones:
            if re.search(patron, desc_norm):
                return rubro
    return "Otros"

print("\n\n🏷️  CLASIFICANDO POR RUBRO...")
canastables["rubro"] = canastables["desc_norm"].apply(clasificar_rubro)

print("\nDistribución por rubro (productos canastables):")
print(canastables["rubro"].value_counts().to_string())

# -----------------------------------------------------------------------------
# 5.5) Vista previa: top productos por rubro
# -----------------------------------------------------------------------------
print("\n\n🔝 TOP 5 PRODUCTOS POR RUBRO (ordenados por cobertura)")
print("=" * 60)

for rubro in ["Bebidas", "Lácteos", "Almacén", "Panadería y harinas",
              "Limpieza", "Higiene personal", "Otros"]:
    sub = (canastables[canastables["rubro"] == rubro]
           .sort_values("n_banderas", ascending=False)
           .head(5))
    if len(sub) > 0:
        print(f"\n--- {rubro} ---")
        print(sub[["id_producto", "descripcion", "marca",
                    "cant_presentacion", "unidad_presentacion",
                    "n_banderas", "precio_mediano", "ratio_p90_p10"]]
              .to_string(index=False))

print(f"\n✅ Paso 5 completo. Variables clave:")
print(f"   - productos_metricas  ({len(productos_metricas):,} filas)")
print(f"   - canastables         ({len(canastables):,} filas)")

🔬 CALCULANDO MÉTRICAS POR PRODUCTO
⏳ Agregando... (puede tardar 1-2 min por el volumen)
✅ Tabla productos_metricas construida: 86,927 productos

📊 DISTRIBUCIÓN DE MÉTRICAS DE COBERTURA Y CALIDAD

Cobertura (n_banderas):
count    86927.000000
mean         3.125450
std          3.059851
min          1.000000
25%          1.000000
50%          3.000000
75%          3.000000
max         21.000000

Cobertura geográfica (n_provincias):
count    86927.000000
mean        12.762341
std          9.061653
min          0.000000
25%          3.000000
50%         13.000000
75%         22.000000
max         24.000000

Consistencia de precio (ratio_p90_p10):
count    86927.000000
mean         1.293986
std         21.584324
min          1.000000
25%          1.000000
50%          1.000000
75%          1.100008
max       3855.356957

🎯 FILTROS APLICADOS
   n_banderas    ≥ 8
   n_provincias  ≥ 5
   ratio p90/p10 ≤ 3.0
   precio_mediano ≥ $50.0

✨ Productos canastables: 7,616 de 86,927 (8.8%)


🏷️  CLASIF

In [16]:
# =============================================================================
# PASO 6: Refinamiento del clasificador y selección de canasta
# =============================================================================
import pandas as pd
import numpy as np
import re
import unicodedata

# -----------------------------------------------------------------------------
# 6.1) Reglas de clasificación mejoradas
# -----------------------------------------------------------------------------
# Cambios vs Paso 5:
#   - Orden reordenado: primero rubros MUY específicos (bebé, alcohol),
#     después los ambiguos. "Crema" cae en Lácteos solo si NO hay "antitrans",
#     "corporal", "dental", etc. en la misma descripción.
#   - Más keywords y abreviaturas típicas de SEPA ("hamb", "crem dent", etc.).
#   - Usamos función con chequeos de exclusión, no regex puros.
# =============================================================================

def normalizar_texto(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return s

# Helper: ¿el texto contiene alguna de estas palabras?
def contiene(texto, palabras):
    return any(p in texto for p in palabras)

def clasificar_rubro_v2(desc_norm):
    """
    Clasificador basado en orden de precedencia y keywords de exclusión.
    Orden: de más específico a más general.
    """
    t = desc_norm  # alias corto

    # --- 1) BEBÉ (muy específico, va primero) ---
    if contiene(t, ["panal", "pañal", "huggies", "pampers", "babysec"]):
        return "Bebé"
    if contiene(t, ["leche"]) and contiene(t, ["infant", "bebe", "nidina", "nan 1", "nan 2"]):
        return "Bebé"

    # --- 2) ALCOHOL ---
    if contiene(t, ["vino ", "vino\t", "tinto", "malbec", "cabernet", "chardonnay",
                     "cerveza", "quilmes", "brahma", "stella", "heineken",
                     "fernet", "whisky", "whiskey", "johnnie walker",
                     "vodka", " gin ", "aperol", "campari", "ron ", "tequila",
                     "aperitivo", "espumante", "champag", "sidra",
                     "vermouth", "vermut", "gancia",
                     "licor", "cynar"]):
        return "Alcohol"

    # --- 3) HIGIENE PERSONAL (chequeamos antes que Limpieza y Lácteos
    #        para no confundir "crema" o "jabon" mal clasificados) ---
    if contiene(t, ["shampoo", "acondicion", "enjuague bucal",
                     "pasta dent", "crem dent", "cepillo dent", "hilo dent",
                     "desodor", "antitrans", "roll on", "rollon",
                     "afeit", "gillette", "rasurad", "espuma afeit",
                     "protec femen", "toall femen", "tampon", "diario larg",
                     "protector diar", "protec diar",
                     "jabon toc", "jabon glic", "jabon liq tocad",
                     "gel intim", "gel int ",
                     "colonia", "perfume"]):
        return "Higiene personal"

    # --- 4) LIMPIEZA (antes que Almacén, porque "jabon" puede ser de ropa) ---
    if contiene(t, ["lavandina", "detergente", "lavavajill",
                     "jab polvo", "jab liq", "jabon polvo", "jabon liq",
                     "jab en polvo", "suavizante", "perfumante", "desinfect",
                     "limpiador", "limpia ", "multiuso",
                     "papel higi", "rollo cocina", "servilleta",
                     "trapo", "trapeador", "esponja", "lampazo",
                     "bolsa consorc", "bolsa residuo"]):
        return "Limpieza"

    # --- 5) LÁCTEOS ---
    if contiene(t, ["yogur", "yogh"]):
        return "Lácteos"
    if contiene(t, ["manteca", "margarina"]):
        return "Lácteos"
    if contiene(t, ["queso", "ricota", "mozzarella", "mozarella", "provolone",
                     "muzza", "cremoso", "rallado"]):
        return "Lácteos"
    if "dulce" in t and "leche" in t:
        return "Lácteos"
    # "leche" solo si no es de bebé (ya filtrado arriba) ni chocolatada
    if contiene(t, ["leche ent", "leche desc", "leche parc", "leche fluid",
                     "leche larga", "leche uat"]):
        return "Lácteos"
    # "crema" como lácteo solo si es "crema de leche" (no corporal, no dental)
    if "crema" in t and contiene(t, ["crema leche", "crema d leche",
                                       "crema d/leche"]):
        return "Lácteos"

    # --- 6) CARNES Y FIAMBRES ---
    if contiene(t, [" carne ", "bife", "asado ", "matambre",
                     "pollo", "pechuga", "pata muslo", "suprema",
                     "cerdo", "bondiola", "panceta",
                     "milanesa", "milan ",
                     "hamb ", "hamburg", "hamburguesa",
                     "salchich", "chorizo", "morcilla",
                     "jamon", " jam ", "salame", "salami", "mortadela",
                     "fiambre", "pate", "paté",
                     "atun", "sardina", "caballa", "anchoa", "salmon"]):
        return "Carnes y fiambres"

    # --- 7) PANADERÍA Y HARINAS ---
    if contiene(t, ["harina", "pan ", " pan\t", "pan lactal",
                     "tostada", "gallet", "galletit", "galleta",
                     "bizcoch", "factura", "medialuna", "criollito",
                     "fideo", "pasta seca", "tallarin", "spaghetti",
                     "polenta", "avena", "cereal", "granola",
                     "arroz"]):
        return "Panadería y harinas"

    # --- 8) ALMACÉN ---
    if contiene(t, ["aceite", "vinagre", "sal fina", "sal grues", "sal entref",
                     " sal ", "azucar", "edulcor",
                     "mayonesa", "ketchup", "mostaza",
                     "salsa", "aderezo",
                     "tomate tr", "tomate peri", "pure tom", "extracto",
                     "lenteja", "poroto", "garbanzo", "legumbre",
                     "conserv", "mermelad",
                     "caldo", "sopa", "gelatina", "flan",
                     "yerba", "te ", " te\t", "cafe", "mate coc"]):
        return "Almacén"

    # --- 9) BEBIDAS (sin alcohol) ---
    if contiene(t, ["gaseos", "cola ", " cola\t", "sprite", "fanta", "pepsi",
                     "manaos", "mirinda", "7up", "7 up",
                     "agua min", "agua sin", "agua con", "agua s/gas",
                     "agua c/gas", "agua natur", "agua saborizada",
                     "isot", "powerade", "gatorade",
                     "jugo ", "nectar", "exprimido", "zuko",
                     "chocolatada", "leche choc", "cindor"]):
        return "Bebidas"

    # --- 10) SNACKS Y GOLOSINAS ---
    if contiene(t, ["alfajor", "chocolate", "bombon", "caramelo",
                     "chupetin", "goma de ", "turron", "barra cereal",
                     "papas frit", "papa frita", "palit salad",
                     "snack", "chizit", "nachos", "pororo", "pochoclo",
                     "mani ", "pasas uva"]):
        return "Snacks y golosinas"

    return "Otros"

print("🏷️  RECLASIFICANDO CON REGLAS MEJORADAS")
print("=" * 60)

# Aplicamos al dataframe canastables (desc_norm ya está calculada del paso 5)
canastables["rubro"] = canastables["desc_norm"].apply(clasificar_rubro_v2)

print("\nDistribución por rubro (nueva clasificación):")
distrib = canastables["rubro"].value_counts()
total = len(canastables)
for rubro, n in distrib.items():
    print(f"   {rubro:<22} {n:>5}  ({100*n/total:>4.1f}%)")

# -----------------------------------------------------------------------------
# 6.2) Herramienta de inspección: ver qué hay en cada rubro
# -----------------------------------------------------------------------------
# Función interactiva para explorar cualquier rubro y detectar mal clasificados.
# =============================================================================

def inspeccionar_rubro(rubro, n=20, sort_by="n_banderas"):
    """Muestra n productos del rubro especificado."""
    sub = (canastables[canastables["rubro"] == rubro]
           .sort_values(sort_by, ascending=False)
           .head(n))
    print(f"\n📋 RUBRO: {rubro}  (mostrando top {min(n, len(sub))} por {sort_by})")
    print("-" * 80)
    print(sub[["id_producto", "descripcion", "marca",
                "cant_presentacion", "unidad_presentacion",
                "n_banderas", "precio_mediano"]]
          .to_string(index=False))

# Mostramos una vista del "Otros" para ver qué sigue sin clasificar
print("\n")
inspeccionar_rubro("Otros", n=30)

# -----------------------------------------------------------------------------
# 6.3) Propuesta de canasta: top-N por rubro con criterios de calidad
# -----------------------------------------------------------------------------
# Lógica: para cada rubro, tomar los N productos con MEJOR cobertura,
# pero con restricción de diversidad de marca (no todos del mismo productor).
# =============================================================================

def armar_canasta(canastables, top_por_rubro=10, max_por_marca=3):
    """
    Selecciona una canasta equilibrada: hasta N productos por rubro,
    con no más de M productos de la misma marca por rubro (para evitar
    que la canasta sea "todo Coca-Cola").
    """
    canasta = []
    for rubro in canastables["rubro"].unique():
        if rubro == "Otros":
            continue  # lo excluimos por ahora, ya lo inspeccionarás manual
        sub = (canastables[canastables["rubro"] == rubro]
               .sort_values(["n_banderas", "n_sucursales"], ascending=False))
        seleccionados = []
        marca_count = {}
        for _, row in sub.iterrows():
            m = row["marca"]
            if marca_count.get(m, 0) >= max_por_marca:
                continue
            seleccionados.append(row)
            marca_count[m] = marca_count.get(m, 0) + 1
            if len(seleccionados) >= top_por_rubro:
                break
        if seleccionados:
            canasta.append(pd.DataFrame(seleccionados))
    return pd.concat(canasta, ignore_index=True) if canasta else pd.DataFrame()

canasta_propuesta = armar_canasta(canastables, top_por_rubro=10, max_por_marca=3)

print("\n\n🛒 CANASTA PROPUESTA (10 por rubro, máx 3 por marca)")
print("=" * 60)
print(f"Total productos en la canasta: {len(canasta_propuesta)}")
print(f"\nDistribución por rubro:")
print(canasta_propuesta["rubro"].value_counts().to_string())

print(f"\nDiversidad de marcas: {canasta_propuesta['marca'].nunique()} marcas distintas")

# Vista completa de la canasta
print("\n\n📋 CANASTA COMPLETA:")
print("=" * 60)
for rubro in canasta_propuesta["rubro"].unique():
    print(f"\n--- {rubro} ---")
    sub = canasta_propuesta[canasta_propuesta["rubro"] == rubro]
    print(sub[["descripcion", "marca", "cant_presentacion",
                "unidad_presentacion", "n_banderas", "precio_mediano"]]
          .to_string(index=False))

print(f"\n✅ Paso 6 completo. Variables clave:")
print(f"   - canastables        ({len(canastables):,} productos candidatos)")
print(f"   - canasta_propuesta  ({len(canasta_propuesta)} productos seleccionados)")
print(f"\n💡 Tip: usá inspeccionar_rubro('NombreRubro', n=30) para explorar.")

🏷️  RECLASIFICANDO CON REGLAS MEJORADAS

Distribución por rubro (nueva clasificación):
   Otros                   2971  (39.0%)
   Almacén                 1085  (14.2%)
   Alcohol                  829  (10.9%)
   Panadería y harinas      493  ( 6.5%)
   Higiene personal         456  ( 6.0%)
   Lácteos                  440  ( 5.8%)
   Limpieza                 407  ( 5.3%)
   Bebidas                  384  ( 5.0%)
   Carnes y fiambres        264  ( 3.5%)
   Snacks y golosinas       189  ( 2.5%)
   Bebé                      98  ( 1.3%)



📋 RUBRO: Otros  (mostrando top 30 por n_banderas)
--------------------------------------------------------------------------------
  id_producto                                    descripcion         marca  cant_presentacion unidad_presentacion  n_banderas  precio_mediano
7790360720115                             PICADILLO DE CARNE         SWIFT              90.00                  GR          20     1279.000000
7790895000997                               